### Load data and do a quick check

In [79]:
import json
import pandas as pd
with open("../data/raw/batdongsan.txt", encoding="utf-8") as file:
    df = pd.DataFrame(json.load(file))

In [76]:
df.head(3)

,Loại hình nhà ở,Diện tích đất,Tổng số tầng,Giấy tờ pháp lý,address,price,city,Số phòng ngủ,Số phòng vệ sinh,Hướng ban công,Hướng cửa chính,Dự án,Tầng số
0,Nhà mặt tiền,65 m²(4.0x16.0),4,Sổ hồng,"Đường Nguyễn Trãi, Phường 7, Quận 5, TP.HCM",48.0,ho-chi-minh,NaN,NaN,NaN,NaN,NaN,NaN
1,Nhà hẻm ngõ,"105 m²(4,6x23,0)",2,Sổ hồng,"1234, Đường Huỳnh Tấn phát, Phường Tân Phú, Qu...",8.2,ho-chi-minh,6 phòng,6 WC,Đông,NaN,NaN,NaN
2,Biệt thự,"1.100 m²(20,0x50,0)",3,Sổ hồng,"105, Đường Trần Văn Kiểu, Phường 10, Quận 6, T...",300.0,ho-chi-minh,8 phòng,10 WC,NaN,NaN,NaN,NaN


In [77]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5414 entries, 0 to 5413
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Loại hình nhà ở   5414 non-null   object
 1   Diện tích đất     5414 non-null   object
 2   Tổng số tầng      3352 non-null   object
 3   Giấy tờ pháp lý   5414 non-null   object
 4   address           5414 non-null   object
 5   price             5414 non-null   object
 6   city              5414 non-null   object
 7   Số phòng ngủ      5008 non-null   object
 8   Số phòng vệ sinh  4631 non-null   object
 9   Hướng ban công    463 non-null    object
 10  Hướng cửa chính   2804 non-null   object
 11  Dự án             110 non-null    object
 12  Tầng số           1 non-null      object
dtypes: object(13)
memory usage: 550.0+ KB


### Basic cleaning (before merging and more preprocessing)
Rename columns

In [80]:
rename_map = {
    "Loại hình nhà ở":"property_type",
    "Diện tích đất":"area",
    "Tổng số tầng":"n_floors",
    "Giấy tờ pháp lý":"legal_docs",
    "Số phòng ngủ":"n_bedrooms",
    "Số phòng vệ sinh":"n_bathrooms",
    "Hướng ban công":"balcony_direction",
    "Hướng cửa chính":"facing_direction",
    "Dự án":"project",
    "Tầng số":"floor_num",
    "city":"city/district"
}
df.rename(rename_map, axis=1, inplace=True)
df.head(2)

,property_type,area,n_floors,legal_docs,address,price,city/district,n_bedrooms,n_bathrooms,balcony_direction,facing_direction,project,floor_num
0,Nhà mặt tiền,65 m²(4.0x16.0),4,Sổ hồng,"Đường Nguyễn Trãi, Phường 7, Quận 5, TP.HCM",48.0,ho-chi-minh,NaN,NaN,NaN,NaN,NaN,NaN
1,Nhà hẻm ngõ,"105 m²(4,6x23,0)",2,Sổ hồng,"1234, Đường Huỳnh Tấn phát, Phường Tân Phú, Qu...",8.2,ho-chi-minh,6 phòng,6 WC,Đông,NaN,NaN,NaN


Remove unusable columns (>50% missing or redundant)

In [81]:
mostly_empty_cols = ['balcony_direction', 'project', 'floor_num']
redundant_cols = ['address']        # only city is enough
df.drop(mostly_empty_cols + redundant_cols, axis=1, inplace=True)

Extract the numeric values for numeric columns

In [ ]:
import re

def extract_numeric(s:str|None, thousands_sep:bool=True) -> float:
    if not isinstance(s,str) or not s:       # Handle np.nan (a float), or empty strings
        return None                        # Return np.nan cuz None is treated like a value

    results = re.search(r"^(\d+.?\d*,?\d*)\D?", s)
    if not results:
        return None

    num_str = results.group(1)
    if thousands_sep:
        num_str = num_str.replace(".","")
    num_str = num_str.replace(",",".")

    return float(num_str)
    
def extract_measuring_unit(s:str|None) -> str:
    if not isinstance(s,str) or not s:
        return None

    results = re.search(r"^\d+.?\d*,?\d*\s*(\D*)", s)
    if not results:
        return None
    else:
        return results.group(1)

def extract_dimensions(area_raw:str):
    if not isinstance(area_raw,str) or not area_raw:
        return None

    results = re.search(r"\((.*)x(.*)\)", area_raw)
    if not results:
        return pd.Series((None, None))
    return pd.Series((
        extract_numeric(results.group(1), thousands_sep=False), 
        extract_numeric(results.group(2), thousands_sep=False)
    ))

In [84]:
df[['dimension_1', 'dimension_2']] = df['area'].apply(extract_dimensions)
df['area_num'] = df['area'].apply(extract_numeric)
df['n_bedrooms_2'] = df['n_bedrooms'].apply(extract_numeric)
df['n_bathrooms_2'] = df['n_bathrooms'].apply(extract_numeric)
df['price_2'] = pd.to_numeric(df['price'], errors='coerce', downcast='float') * 1000    # to million
df['n_floors_2'] = pd.to_numeric(df['n_floors'], errors='coerce')

df[['area', 'area_num', 'dimension_1', 'dimension_2', 
    'n_bedrooms', 'n_bedrooms_2', 'n_bathrooms', 'n_bathrooms_2', 
    'price', 'price_2', 
    'n_floors', 'n_floors_2']].sample(10)

,area,area_num,dimension_1,dimension_2,n_bedrooms,n_bedrooms_2,n_bathrooms,n_bathrooms_2,price,price_2,n_floors,n_floors_2
384,"72 m²(4,0x18,0)",72.0,4.0,18.0,2 phòng,2.0,2 WC,2.0,4.5,4500.0,2,2.0
4199,"48,5 m²(5,3x9,0)",48.5,5.3,9.0,2 phòng,2.0,2 WC,2.0,1.48,1480.0,2,2.0
3571,"163 m²(6,0x27,0)",163.0,6.0,27.0,2 phòng,2.0,2 WC,2.0,1.85,1850.0,NaN,NaN
2,"1.100 m²(20,0x50,0)",1100.0,20.0,50.0,8 phòng,8.0,10 WC,10.0,300.0,300000.0,3,3.0
593,162 m²(8.0x21.0),162.0,8.0,21.0,15 phòng,15.0,16 WC,16.0,52.0,52000.0,7,7.0
733,"72 m²(4,0x18,0)",72.0,4.0,18.0,6 phòng,6.0,5 WC,5.0,15.0,15000.0,4,4.0
1369,"60 m²(4,0x15,0)",60.0,4.0,15.0,3 phòng,3.0,2 WC,2.0,6.2,6200.0,1,1.0
1751,"70 m²(4,3x15,0)",70.0,4.3,15.0,3 phòng,3.0,2 WC,2.0,3.09,3090.0,NaN,NaN
2322,"73 m²(4,0x18,5)",73.0,4.0,18.5,3 phòng,3.0,2 WC,2.0,3.85,3850.0,2,2.0
3787,"110 m² (5,0x22,0)",110.0,5.0,22.0,2 phòng,2.0,1 WC,1.0,2.1,2100.0,1,1.0


In [85]:
df[['area', 'area_num', 'dimension_1', 'dimension_2', 
    'n_bedrooms', 'n_bedrooms_2', 'n_bathrooms', 'n_bathrooms_2', 
    'price', 'price_2', 
    'n_floors', 'n_floors_2'
]].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5414 entries, 0 to 5413
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   area           5414 non-null   object 
 1   area_num       5414 non-null   float64
 2   dimension_1    4692 non-null   float64
 3   dimension_2    4692 non-null   float64
 4   n_bedrooms     5008 non-null   object 
 5   n_bedrooms_2   5008 non-null   float64
 6   n_bathrooms    4631 non-null   object 
 7   n_bathrooms_2  4631 non-null   float64
 8   price          5414 non-null   object 
 9   price_2        5406 non-null   float32
 10  n_floors       3352 non-null   object 
 11  n_floors_2     3352 non-null   float64
dtypes: float32(1), float64(6), object(5)
memory usage: 486.5+ KB


In [86]:
df.loc[df.price_2.isna()]

,property_type,area,n_floors,legal_docs,price,city/district,n_bedrooms,n_bathrooms,facing_direction,dimension_1,dimension_2,area_num,n_bedrooms_2,n_bathrooms_2,price_2,n_floors_2
13,Nhà mặt tiền,40 m²,2,Sổ đỏ,thỏa thuận,ho-chi-minh,NaN,NaN,NaN,NaN,NaN,40.0,NaN,NaN,NaN,2.0
97,Nhà mặt tiền,"49,7 m²(4,0x12,0)",NaN,Sổ hồng,thỏa thuận,ho-chi-minh,3 phòng,1 WC,NaN,4.0,12.0,49.7,3.0,1.0,NaN,NaN
280,Nhà hẻm ngõ,"315 m²(8,5x16,4)",2,Sổ hồng,thỏa thuận,ho-chi-minh,NaN,NaN,NaN,8.5,16.4,315.0,NaN,NaN,NaN,2.0
1023,Nhà mặt tiền,80 m²,NaN,Sổ đỏ,thỏa thuận,ha-noi,NaN,NaN,NaN,NaN,NaN,80.0,NaN,NaN,NaN,NaN
1102,Nhà hẻm ngõ,43 m²,NaN,Sổ đỏ,thỏa thuận,ha-noi,NaN,NaN,NaN,NaN,NaN,43.0,NaN,NaN,NaN,NaN
1114,Nhà hẻm ngõ,80 m²,5,Sổ đỏ,thỏa thuận,ha-noi,7 phòng,NaN,NaN,NaN,NaN,80.0,7.0,NaN,NaN,5.0
4026,Biệt thự,"56 m²(4,0x14,0)",1,Sổ hồng,thỏa thuận,binh-thuan,2 phòng,2 WC,Tây,4.0,14.0,56.0,2.0,2.0,NaN,1.0
5405,Biệt thự,180 m²,2,Sổ đỏ,thỏa thuận,vinh-phuc,NaN,NaN,NaN,NaN,NaN,180.0,NaN,NaN,NaN,2.0


Observation:
- Non-null counts of `price_2` is lower than the original `price`: "Negotiable" prices 

### Export the file with extracted features

In [87]:
df_final = df[['property_type', 'price_2', 'area_num', 'n_bedrooms_2', 'n_bathrooms_2',
                'legal_docs', 'city/district', 'facing_direction', 'dimension_1', 'n_floors_2']]
df_final.to_csv('../data/interim/muaban_net.csv')

In [88]:
reload_df_test = pd.read_csv('../data/interim/muaban_net.csv')
reload_df_test.sample(10)

,Unnamed: 0,property_type,price_2,area_num,n_bedrooms_2,n_bathrooms_2,legal_docs,city/district,facing_direction,dimension_1,n_floors_2
315,315,Nhà hẻm ngõ,5800.0,66.0,NaN,NaN,Sổ hồng,ho-chi-minh,NaN,NaN,NaN
718,718,Nhà hẻm ngõ,4850.0,42.0,3.0,3.0,Giấy tờ hợp lệ,ho-chi-minh,Tây Bắc,3.6,4.0
1030,1030,Nhà hẻm ngõ,7100.0,35.0,NaN,NaN,Sổ đỏ,ha-noi,NaN,NaN,3.0
444,444,Nhà hẻm ngõ,1050.0,175.0,2.0,NaN,Sổ hồng,ho-chi-minh,NaN,6.0,NaN
657,657,Nhà hẻm ngõ,9000.0,54.0,6.0,6.0,Giấy tờ hợp lệ,ho-chi-minh,NaN,4.4,3.0
739,739,Nhà mặt tiền,14000.0,80.0,6.0,6.0,Giấy tờ hợp lệ,ho-chi-minh,NaN,5.0,5.0
380,380,Nhà hẻm ngõ,4550.0,48.0,4.0,2.0,Giấy tờ hợp lệ,ho-chi-minh,Đông Bắc,4.0,2.0
4476,4476,Nhà mặt tiền,19950.0,149.0,4.0,3.0,Giấy tờ hợp lệ,quang-binh,NaN,8.0,3.0
3068,3068,Nhà hẻm ngõ,1500.0,90.0,3.0,2.0,Hợp đồng mua bán,dak-lak,Đông Nam,9.0,2.0
1747,1747,Nhà mặt tiền,3990.0,64.0,3.0,3.0,Giấy tờ hợp lệ,can-tho,Đông Nam,4.0,2.0


In [89]:
reload_df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5414 entries, 0 to 5413
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0        5414 non-null   int64  
 1   property_type     5414 non-null   object 
 2   price_2           5406 non-null   float64
 3   area_num          5414 non-null   float64
 4   n_bedrooms_2      5008 non-null   float64
 5   n_bathrooms_2     4631 non-null   float64
 6   legal_docs        5414 non-null   object 
 7   city/district     5414 non-null   object 
 8   facing_direction  2804 non-null   object 
 9   dimension_1       4692 non-null   float64
 10  n_floors_2        3352 non-null   float64
dtypes: float64(6), int64(1), object(4)
memory usage: 465.4+ KB
